*This is the notebook for the training phase of this example*

## Convolutional Neural Network Demonstration (For MNIST)
**Goal:** Recognise the digit drawn on the image, shooting for 98%+ accuracy on Kaggle.

**Strategy:**
1. Architecture: 784 - (28 x 28) - (7 x 7) - 49 - 32 - 16 - 10
2. We do some basic preprocessing of the data, such as centering the image, and normalising the brightest pixel to 1.0
2. We have a data set of 42000 images, we will use the first 1000 images to quickly validate the accuracy of our program, and the remaining will be used to train.
3. We use Adam optimizer to optimize.
4. We save our training result to a `.cache` file.

This library works similarly to Pytorch, and I'm proud of that.

### 1. Import and Config

In [2]:
import math
import lktorch as lk
import os
import pandas as pd


CURRENT_DIR = os.getcwd()
WEIGHT_PATH = os.path.join(CURRENT_DIR, "training_data", "model_weight.cache")
DATA_PATH = os.path.join(CURRENT_DIR, "training_data", "train.csv")

#### Unrelated. Testing

In [3]:

bro = lk.Zeros([2, 784])
tenso = bro.Reshape([28, 28])
print(tenso.dimension())
tenso = lk.Slice(tenso, [1, 1], [26, 26])
print(tenso.dimension())
tenso = lk.Unfold(tenso, [3, 3])
print(tenso.dimension())

[2, 28, 28]
[2, 26, 26]
[2, 24, 24, 3, 3]


### 2. Generate and preprocess

In [4]:
input = pd.read_csv(DATA_PATH)
row_count, column_count = input.shape

input_dataset = []
output_dataset = []
for i in range(row_count):
    input_row = input.loc[i].to_numpy()

    current_output = input_row[0]
    current_row = [0] * 10
    current_row[current_output] = 1
    output_dataset.append(current_row)
    input_dataset.append(input_row[1:])


def preprocess_row(current_row):
    result_row = [0] * 784
    moment_x = 0
    moment_y = 0
    sum = 0
    for i in range(0, 784):
        x = i // 28
        y = i % 28
        moment_x += x * current_row[i]
        moment_y += y * current_row[i]
        sum += current_row[i]
    centroid_x = moment_x // sum
    centroid_y = moment_y // sum

    centroid_x = 14 - centroid_x
    centroid_y = 14 - centroid_y

    for i in range(0, 784):
        x = i // 28
        y = i % 28

        x += centroid_x
        y += centroid_y

        if (x >= 0 and x < 28 and y >= 0 and y < 28):
            result_row[x * 28 + y] = current_row[i]

    ma = 0
    for i in result_row:
        ma = max(ma, i)
    for i in range(0, 784):
        result_row[i] = result_row[i] / ma

    return result_row

for i in range(len(input_dataset)):
    input_dataset[i] = preprocess_row(input_dataset[i])

print("Done fetching data!")

Done fetching data!


### 3. Prepare for the training

In [5]:
input_tensor = lk.Tensor(input_dataset)
output_tensor = lk.Tensor(output_dataset)

train_input_tensor = input_tensor.Slice([1000, 0], [41999, 783])
train_output_tensor = output_tensor.Slice([1000, 0], [41999, 9])

eval_input_tensor = input_tensor.Slice([0, 0], [999, 783])
eval_output_tensor = output_tensor.Slice([0, 0], [999, 9])


def calculate_loss(model, input_data, output_data, LossFunction):
    tenso = model(input_data)
    tenso = LossFunction(tenso, output_data)
    return lk.Mean(tenso)


### 4. Defining the model

In [6]:
model = lk.Sequential([
    lk.Reshape_Layer([28, 28]),
    lk.Slice_Layer([2, 2], [23, 23]), # (22, 22)
    lk.Unfold_Layer([3, 3]), # (20, 20, 3, 3)
    lk.Conv2D([3, 3], [10]), # (20, 20, 10)
    lk.reLU_Layer(),
    lk.PermuteDimension_Layer([2, 0, 1]), # (10, 20, 20)
    lk.Unfold_Layer([2, 2], [2, 2]), # (10, 10, 10, 2, 2) 
    lk.MaxPool_Layer(),
    lk.MaxPool_Layer(), # (10, 10, 10)
    lk.Reshape_Layer([1000]), # (1000)
    
    lk.LinearLayer(1000, 40),
    lk.reLU_Layer(),
    lk.Dropout_Layer(0.2),
    lk.LinearLayer(40, 40),
    lk.reLU_Layer(),
    lk.Dropout_Layer(0.2),
    lk.LinearLayer(40, 10),
    lk.Dropout_Layer(0.2),
    lk.ExpNormalize_Layer()
])


lospollos = model(input_tensor.Slice([0, 0], [1, 783]))
print(lospollos)
cur_loss = calculate_loss(model, eval_input_tensor, eval_output_tensor, lk.MSELoss).accessA([])
print(cur_loss)

((0, 0, 4.91561e-21, 0, 0, 0, 0, 1, 0, 0), (0, 5.15538e-41, 0, 0, 0, 0, 0, 3.3491e-43, 1, 0))
0.1798349916934967


### 5. Training and saving to file

In [ ]:

lr = 0.01

optimizer = lk.GradientDescent(lr)
optimizer.add_parameter(model.get_parameters())
best = 1e18
for epoch in range(1, 501):
    BATCH = 42
    l = 0
    lim = train_input_tensor.dimension()[0]
    while (l < lim):
        optimizer.zero_gradient()

        r = min(l + BATCH, lim) - 1

        input_batch = train_input_tensor.Slice([l, 0], [r, 783])
        tenso = model(input_batch)
        tenso = lk.Mean(lk.CrossEntropyLoss(tenso, train_output_tensor.Slice([l, 0], [r, 9])))

        tenso.backward()
        optimizer.step()
        l += BATCH
    print("epoch:", epoch)
    cur_loss = calculate_loss(model, eval_input_tensor, eval_output_tensor, lk.CrossEntropyLoss).accessA([])
    print("loss: ", cur_loss)
    if (cur_loss < best):
        best = cur_loss
        lk.WriteFile(model.get_state_dict(), WEIGHT_PATH)
        print("File saved successfully!")


    lr *= 0.96
    optimizer.set_learning_rate(lr)


epoch: 1
loss:  2.4373226165771484
File saved successfully!
epoch: 2
loss:  2.27421236038208
File saved successfully!
epoch: 3
loss:  2.1038148403167725
File saved successfully!
epoch: 4
loss:  2.039623975753784
File saved successfully!
epoch: 5
loss:  1.9345067739486694
File saved successfully!
epoch: 6
loss:  1.7984877824783325
File saved successfully!
epoch: 7
loss:  1.6748584508895874
File saved successfully!
epoch: 8
loss:  1.5948777198791504
File saved successfully!
epoch: 9
loss:  1.523541808128357
File saved successfully!
epoch: 10
loss:  1.4250417947769165
File saved successfully!
epoch: 11
loss:  1.3652435541152954
File saved successfully!
epoch: 12
loss:  1.4041775465011597
epoch: 13
loss:  1.3092389106750488
File saved successfully!
epoch: 14
loss:  1.3302730321884155
epoch: 15
loss:  1.2618111371994019
File saved successfully!
epoch: 16
loss:  1.264851689338684
epoch: 17
loss:  1.2047882080078125
File saved successfully!
epoch: 18
loss:  1.2208396196365356
epoch: 19
loss: 

### 6. Verify model training result

In [7]:
def get_label(tensor):
    label = []
    sz = tensor.dimension()
    tensor = lk.MaxIndex(tensor)
    label = tensor.numpy()
    return label

def calculate_fault(tensor, label):
    correct = 0
    my_label = get_label(tensor)
    for i in range(len(label)):
        if (my_label[i] == label[i]):
            correct += 1
    return correct / len(label)

label = []
for i in range(1000):
    input_row = input.loc[i].to_numpy()
    label.append(input_row[0])


lk.deactivate_backprop()

model.load_state_dict(lk.ReadFile(WEIGHT_PATH))
tenso = model(eval_input_tensor)
print("Accuracy: ", calculate_fault(tenso, label))


print("Sanity check: ")
print("Accuracy of output dataset: ", calculate_fault(output_tensor, label))

Accuracy:  0.942
Sanity check: 
Accuracy of output dataset:  1.0
